# MOTIF Preprocessing Timing Notebook

This notebook measures the time and memory consumption of MOTIF's preprocessing pipeline on the WN18RR dataset.

## Stages Measured:
1. Dataset Loading
2. Relation Graph Construction
3. Hypergraph Construction (split into sub-stages)

In [30]:
import torch
torch.__version__

'2.10.0+cu128'

In [ ]:
!pip install torch-sparse -f https://pytorch-geometric.com/whl/torch-2.10.0+cu128.html
!pip install torch-scatter -f https://pytorch-geometric.com/whl/torch-2.10.0+cu128.html
!pip install torch-geometric

Looking in links: https://pytorch-geometric.com/whl/torch-2.10.0+cu128.html
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py

In [1]:
# @title 2. Imports
import os
import time
import psutil
from typing import Optional, Dict, Any

import torch
import torch.nn.functional as F
from torch import Tensor
from torch_scatter import scatter
from torch_sparse import SparseTensor
from torch_geometric.data import Data
from torch_geometric.datasets import WordNet18RR
from torch_geometric.utils import coalesce

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


In [2]:
# @title 3. Timing Utilities

class Timer:
    def __init__(self):
        self.results = {}

    def time(self, name: str):
        return TimerContext(self, name)

    def add_result(self, name: str, elapsed: float, memory_delta: float):
        self.results[name] = {
            "time_s": elapsed,
            "memory_mb": memory_delta
        }

    def summary(self):
        print("\n" + "=" * 70)
        print("TIMING SUMMARY")
        print("=" * 70)
        print(f"{'Stage':<40} {'Time (s)':>12} {'Memory (MB)':>15}")
        print("-" * 70)

        total_time = 0
        total_mem = 0

        for name, data in self.results.items():
            print(f"{name:<40} {data['time_s']:>12.3f} {data['memory_mb']:>+15.1f}")
            total_time += data['time_s']
            total_mem += data['memory_mb']

        print("-" * 70)
        print(f"{'TOTAL':<40} {total_time:>12.3f} {total_mem:>+15.1f}")
        print("=" * 70)


class TimerContext:
    def __init__(self, timer: Timer, name: str):
        self.timer = timer
        self.name = name

    def __enter__(self):
        if device == "cuda":
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        self.start_time = time.perf_counter()
        self.start_mem = psutil.Process().memory_info().rss / 1024 / 1024
        return self

    def __exit__(self, *args):
        if device == "cuda":
            torch.cuda.synchronize()
        end_time = time.perf_counter()
        end_mem = psutil.Process().memory_info().rss / 1024 / 1024

        self.timer.add_result(
            self.name,
            end_time - self.start_time,
            end_mem - self.start_mem
        )


timer = Timer()

In [3]:
# @title 4. Build Relation Graph

def build_relation_graph(data: Data) -> Data:
    """Build graph where nodes are relations, edges indicate shared entities."""
    edge_index = data.edge_index
    edge_type = data.edge_type
    num_nodes = data.num_nodes
    num_relations = data.num_relations

    heads = edge_index[0]
    tails = edge_index[1]

    Eh = SparseTensor(
        row=heads, col=edge_type,
        value=torch.ones_like(heads, dtype=torch.float),
        sparse_sizes=(num_nodes, num_relations)
    )

    Et = SparseTensor(
        row=tails, col=edge_type,
        value=torch.ones_like(tails, dtype=torch.float),
        sparse_sizes=(num_nodes, num_relations)
    )

    Ahh = Eh.t() @ Eh
    Att = Et.t() @ Et
    Aht = Eh.t() @ Et
    Ath = Et.t() @ Eh

    rel_edges = []
    rel_types = []

    for adj, rtype in [(Ahh, 0), (Att, 1), (Aht, 2), (Ath, 3)]:
        row, col, _ = adj.coo()
        mask = row != col
        row, col = row[mask], col[mask]
        rel_edges.append(torch.stack([row, col], dim=0))
        rel_types.append(torch.full((row.size(0),), rtype, dtype=torch.long, device=row.device))

    data.relation_graph = Data(
        edge_index=torch.cat(rel_edges, dim=1),
        edge_type=torch.cat(rel_types, dim=0),
        num_nodes=num_relations,
        num_relations=4
    )
    return data

In [4]:
# @title 5. Build Relation Hypergraph

def build_relation_hypergraph(data: Data) -> Data:
    """Build hypergraph encoding 2-path and 3path motifs."""
    edge_index = data.edge_index
    edge_type = data.edge_type
    num_nodes = data.num_nodes
    num_relations = data.num_relations
    device = edge_index.device

    heads = edge_index[0]
    tails = edge_index[1]

    Eh = SparseTensor(
        row=heads, col=edge_type,
        value=torch.ones_like(heads, dtype=torch.float),
        sparse_sizes=(num_nodes, num_relations)
    )

    Et = SparseTensor(
        row=tails, col=edge_type,
        value=torch.ones_like(tails, dtype=torch.float),
        sparse_sizes=(num_nodes, num_relations)
    )

    Ahh = Eh.t() @ Eh
    Att = Et.t() @ Et
    Aht = Eh.t() @ Et

    rel_edges = []
    rel_types = []

    for adj, rtype in [(Ahh, 0), (Att, 1), (Aht, 2)]:
        row, col, _ = adj.coo()
        mask = row != col
        row, col = row[mask], col[mask]
        rel_edges.append(torch.stack([row, col], dim=0))
        rel_types.append(torch.full((row.size(0),), rtype, dtype=torch.long, device=device))

    three_paths = {3: [], 4: [], 5: [], 6: []}

    edges_list = torch.stack([heads, tails, edge_type], dim=1).tolist()

    outgoing = {}
    incoming = {}

    for h, t, r in edges_list:
        if h not in outgoing:
            outgoing[h] = []
        outgoing[h].append((r, t))
        if t not in incoming:
            incoming[t] = []
        incoming[t].append((h, r))

    for h1, t1, r1 in edges_list:
        if t1 in outgoing:
            for r2, t2 in outgoing[t1]:
                if t2 in incoming:
                    for h3, r3 in incoming[t2]:
                        three_paths[3].append((r1, r3))
                if t2 in outgoing:
                    for r3, t3 in outgoing[t2]:
                        three_paths[4].append((r1, r3))

    for h1, t1, r1 in edges_list:
        if h1 in incoming:
            for h2, r2 in incoming[h1]:
                if h2 in outgoing:
                    for r3, t3 in outgoing[h2]:
                        three_paths[5].append((r1, r3))
                if h2 in incoming:
                    for h3, r3 in incoming[h2]:
                        three_paths[6].append((r1, r3))

    for ptype, paths in three_paths.items():
        if paths:
            paths_t = torch.tensor(paths, dtype=torch.long, device=device).t()
            rel_edges.append(paths_t)
            rel_types.append(torch.full((len(paths),), ptype, dtype=torch.long, device=device))

    if rel_edges:
        hyper_edge_index = torch.cat(rel_edges, dim=1)
        hyper_edge_type = torch.cat(rel_types, dim=0)
    else:
        hyper_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        hyper_edge_type = torch.empty(0, dtype=torch.long, device=device)

    data.relation_hypergraph = Data(
        edge_index=hyper_edge_index,
        edge_type=hyper_edge_type,
        num_nodes=num_relations,
        num_relations=7
    )

    return data

In [ ]:
# @title 5.5 Build Relation Hypergraph with Sampling (Python Loop Version)

def build_relation_hypergraph_sampled_python(data: Data, sample_prob: float = 1.0, seed: int = None) -> Data:
    """
    Build hypergraph with proportional stratified sampling of 3-path motifs.
    Uses Python loops (same as build_relation_hypergraph) but samples during enumeration.
    
    Args:
        data: PyG Data object
        sample_prob: Probability of keeping each discovered 3-path motif (0 < sample_prob <= 1)
        seed: Random seed for reproducibility
    """
    import random
    
    if sample_prob <= 0:
        raise ValueError(f"sample_prob must be > 0, got {sample_prob}")
    
    if seed is not None:
        random.seed(seed)
        torch.manual_seed(seed)
    
    edge_index = data.edge_index
    edge_type = data.edge_type
    num_nodes = data.num_nodes
    num_relations = data.num_relations
    device = edge_index.device

    heads = edge_index[0]
    tails = edge_index[1]

    # 2-path motifs (no sampling needed - they're tiny)
    Eh = SparseTensor(
        row=heads, col=edge_type,
        value=torch.ones_like(heads, dtype=torch.float),
        sparse_sizes=(num_nodes, num_relations)
    )
    Et = SparseTensor(
        row=tails, col=edge_type,
        value=torch.ones_like(tails, dtype=torch.float),
        sparse_sizes=(num_nodes, num_relations)
    )

    Ahh = Eh.t() @ Eh
    Att = Et.t() @ Et
    Aht = Eh.t() @ Et

    rel_edges = []
    rel_types = []

    for adj, rtype in [(Ahh, 0), (Att, 1), (Aht, 2)]:
        row, col, _ = adj.coo()
        mask = row != col
        row, col = row[mask], col[mask]
        rel_edges.append(torch.stack([row, col], dim=0))
        rel_types.append(torch.full((row.size(0),), rtype, dtype=torch.long, device=device))

    # 3-path motifs with sampling
    three_paths = {3: [], 4: [], 5: [], 6: []}
    edges_list = torch.stack([heads, tails, edge_type], dim=1).tolist()

    outgoing = {}
    incoming = {}

    for h, t, r in edges_list:
        if h not in outgoing:
            outgoing[h] = []
        outgoing[h].append((r, t))
        if t not in incoming:
            incoming[t] = []
        incoming[t].append((h, r))

    # Sample during enumeration to save memory
    if sample_prob >= 1.0:
        # No sampling - keep all
        for h1, t1, r1 in edges_list:
            if t1 in outgoing:
                for r2, t2 in outgoing[t1]:
                    if t2 in incoming:
                        for h3, r3 in incoming[t2]:
                            three_paths[3].append((r1, r3))
                    if t2 in outgoing:
                        for r3, t3 in outgoing[t2]:
                            three_paths[4].append((r1, r3))

        for h1, t1, r1 in edges_list:
            if h1 in incoming:
                for h2, r2 in incoming[h1]:
                    if h2 in outgoing:
                        for r3, t3 in outgoing[h2]:
                            three_paths[5].append((r1, r3))
                    if h2 in incoming:
                        for h3, r3 in incoming[h2]:
                            three_paths[6].append((r1, r3))
    else:
        # Sample each discovered motif with probability sample_prob
        for h1, t1, r1 in edges_list:
            if t1 in outgoing:
                for r2, t2 in outgoing[t1]:
                    if t2 in incoming:
                        for h3, r3 in incoming[t2]:
                            if random.random() < sample_prob:
                                three_paths[3].append((r1, r3))
                    if t2 in outgoing:
                        for r3, t3 in outgoing[t2]:
                            if random.random() < sample_prob:
                                three_paths[4].append((r1, r3))

        for h1, t1, r1 in edges_list:
            if h1 in incoming:
                for h2, r2 in incoming[h1]:
                    if h2 in outgoing:
                        for r3, t3 in outgoing[h2]:
                            if random.random() < sample_prob:
                                three_paths[5].append((r1, r3))
                    if h2 in incoming:
                        for h3, r3 in incoming[h2]:
                            if random.random() < sample_prob:
                                three_paths[6].append((r1, r3))

    for ptype, paths in three_paths.items():
        if paths:
            paths_t = torch.tensor(paths, dtype=torch.long, device=device).t()
            rel_edges.append(paths_t)
            rel_types.append(torch.full((len(paths),), ptype, dtype=torch.long, device=device))

    if rel_edges:
        hyper_edge_index = torch.cat(rel_edges, dim=1)
        hyper_edge_type = torch.cat(rel_types, dim=0)
    else:
        hyper_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
        hyper_edge_type = torch.empty(0, dtype=torch.long, device=device)

    data.relation_hypergraph = Data(
        edge_index=hyper_edge_index,
        edge_type=hyper_edge_type,
        num_nodes=num_relations,
        num_relations=7
    )

    return data

In [5]:
# @title 6. Load WN18RR

In [5]:
# @title 6. Load WN18RR

def load_wn18rr(root: str, device: str = "cpu") -> tuple:
    """Load WN18RR and convert to MOTIF format."""
    dataset = WordNet18RR(root=root)
    data = dataset.data

    num_nodes = int(data.edge_index.max()) + 1
    num_relations = int(data.edge_type.max()) + 1

    train_mask = data.train_mask if hasattr(data, 'train_mask') else torch.ones(data.edge_index.size(1), dtype=torch.bool)
    val_mask = getattr(data, 'val_mask', torch.zeros(data.edge_index.size(1), dtype=torch.bool))
    test_mask = getattr(data, 'test_mask', torch.zeros(data.edge_index.size(1), dtype=torch.bool))

    train_idx = data.edge_index[:, train_mask]
    train_type = data.edge_type[train_mask]

    edge_index = torch.cat([train_idx, train_idx.flip(0)], dim=-1)
    edge_type = torch.cat([train_type, train_type + num_relations])

    train_data = Data(
        edge_index=edge_index, edge_type=edge_type,
        num_nodes=num_nodes,
        target_edge_index=train_idx, target_edge_type=train_type,
        num_relations=num_relations * 2
    ).to(device)

    val_idx = data.edge_index[:, val_mask]
    val_type = data.edge_type[val_mask]
    val_data = Data(
        edge_index=edge_index, edge_type=edge_type,
        num_nodes=num_nodes,
        target_edge_index=val_idx, target_edge_type=val_type,
        num_relations=num_relations * 2
    ).to(device)

    test_idx = data.edge_index[:, test_mask]
    test_type = data.edge_type[test_mask]
    test_data = Data(
        edge_index=edge_index, edge_type=edge_type,
        num_nodes=num_nodes,
        target_edge_index=test_idx, target_edge_type=test_type,
        num_relations=num_relations * 2
    ).to(device)

    stats = {
        "num_nodes": num_nodes,
        "num_edges": edge_index.size(1),
        "num_relations": num_relations * 2,
        "num_train": train_idx.size(1),
        "num_val": val_idx.size(1),
        "num_test": test_idx.size(1),
    }

    return train_data, val_data, test_data, stats

In [6]:
# @title 7. Run Preprocessing with Timing

root = "/content/wn18rr"

print("=" * 70)
print("MOTIF Preprocessing on WN18RR")
print("=" * 70)

# Stage 1: Load
with timer.time("1. Dataset Loading"):
    train_data, val_data, test_data, stats = load_wn18rr(root, device)
print(f"Nodes: {stats['num_nodes']}, Edges: {stats['num_edges']}, Rels: {stats['num_relations']}")

# Stage 2: Relation graph
with timer.time("2. Relation Graph Construction"):
    train_data = build_relation_graph(train_data)
print(f"Relation graph edges: {train_data.relation_graph.edge_index.size(1)}")

# Stage 3: Hypergraph
with timer.time("3. Hypergraph Construction"):
    train_data = build_relation_hypergraph(train_data)

if hasattr(train_data, 'relation_hypergraph'):
    hg = train_data.relation_hypergraph
    print(f"Hypergraph edges: {hg.edge_index.size(1)}")

    type_counts = torch.bincount(hg.edge_type, minlength=7)
    type_names = ['hh', 'tt', 'ht', 'tfh', 'tft', 'hft', 'hfh']
    print("\nEdges per type:")
    for i, name in enumerate(type_names):
        if i < len(type_counts):
            print(f"  {name}: {type_counts[i].item()}")

# Summary
timer.summary()

MOTIF Preprocessing on WN18RR


/tmp/ipykernel_16931/979614783.py:6: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  data = dataset.data


Nodes: 40943, Edges: 173670, Rels: 22
Relation graph edges: 1432
Hypergraph edges: 96517810

Edges per type:
  hh: 358
  tt: 358
  ht: 358
  tfh: 24129184
  tft: 24129184
  hft: 24129184
  hfh: 24129184

TIMING SUMMARY
Stage                                        Time (s)     Memory (MB)
----------------------------------------------------------------------
1. Dataset Loading                              0.023           +16.1
2. Relation Graph Construction                  0.188          +250.5
3. Hypergraph Construction                     50.646           +10.9
----------------------------------------------------------------------
TOTAL                                          50.858          +277.6


In [9]:
# @title 8. Optional: Detailed Hypergraph Timing

def build_relation_hypergraph_detailed_timing(data: Data, timer: Timer) -> Data:
    """Build hypergraph with detailed substage timing."""
    edge_index = data.edge_index
    edge_type = data.edge_type
    num_nodes = data.num_nodes
    num_relations = data.num_relations
    device = edge_index.device

    heads = edge_index[0]
    tails = edge_index[1]

    # Stage 3a: Build incidence
    with timer.time("3a. Build incidence tensors"):
        Eh = SparseTensor(
            row=heads, col=edge_type,
            value=torch.ones_like(heads, dtype=torch.float),
            sparse_sizes=(num_nodes, num_relations)
        )
        Et = SparseTensor(
            row=tails, col=edge_type,
            value=torch.ones_like(tails, dtype=torch.float),
            sparse_sizes=(num_nodes, num_relations)
        )

    # Stage 3b: 2-path motifs
    with timer.time("3b. 2-path motifs (hh, tt, ht)"):
        Ahh = Eh.t() @ Eh
        Att = Et.t() @ Et
        Aht = Eh.t() @ Et

        rel_edges = []
        rel_types = []

        for adj, rtype in [(Ahh, 0), (Att, 1), (Aht, 2)]:
            row, col, _ = adj.coo()
            mask = row != col
            row, col = row[mask], col[mask]
            rel_edges.append(torch.stack([row, col], dim=0))
            rel_types.append(torch.full((row.size(0),), rtype, dtype=torch.long, device=device))

    # Stage 3c: Prepare 3-path structures
    with timer.time("3c. 3-path prep (build adjacency)"):
        edges_list = torch.stack([heads, tails, edge_type], dim=1).tolist()
        outgoing = {}
        incoming = {}

        for h, t, r in edges_list:
            if h not in outgoing:
                outgoing[h] = []
            outgoing[h].append((r, t))
            if t not in incoming:
                incoming[t] = []
            incoming[t].append((h, r))

    # Stage 3d: Compute 3-paths
    with timer.time("3d. 3-path motifs (tfh, tft, hft, hfh)"):
        three_paths = {3: [], 4: [], 5: [], 6: []}

        for h1, t1, r1 in edges_list:
            if t1 in outgoing:
                for r2, t2 in outgoing[t1]:
                    if t2 in incoming:
                        for h3, r3 in incoming[t2]:
                            three_paths[3].append((r1, r3))
                    if t2 in outgoing:
                        for r3, t3 in outgoing[t2]:
                            three_paths[4].append((r1, r3))

        for h1, t1, r1 in edges_list:
            if h1 in incoming:
                for h2, r2 in incoming[h1]:
                    if h2 in outgoing:
                        for r3, t3 in outgoing[h2]:
                            three_paths[5].append((r1, r3))
                    if h2 in incoming:
                        for h3, r3 in incoming[h2]:
                            three_paths[6].append((r1, r3))

        for ptype, paths in three_paths.items():
            if paths:
                paths_t = torch.tensor(paths, dtype=torch.long, device=device).t()
                rel_edges.append(paths_t)
                rel_types.append(torch.full((len(paths),), ptype, dtype=torch.long, device=device))

    # Stage 3e: Combine
    with timer.time("3e. Combine hypergraph"):
        if rel_edges:
            hyper_edge_index = torch.cat(rel_edges, dim=1)
            hyper_edge_type = torch.cat(rel_types, dim=0)
        else:
            hyper_edge_index = torch.empty((2, 0), dtype=torch.long, device=device)
            hyper_edge_type = torch.empty(0, dtype=torch.long, device=device)

        data.relation_hypergraph = Data(
            edge_index=hyper_edge_index,
            edge_type=hyper_edge_type,
            num_nodes=num_relations,
            num_relations=7
        )

    return data

# Run detailed timing
print("\n" + "=" * 70)
print("DETAILED HYPERGRAPH TIMING")
print("=" * 70)

detailed_timer = Timer()
train_data2, _, _, stats2 = load_wn18rr(root, device)
train_data2 = build_relation_graph(train_data2)
train_data2 = build_relation_hypergraph_detailed_timing(train_data2, detailed_timer)

detailed_timer.summary()


DETAILED HYPERGRAPH TIMING


/tmp/ipykernel_7159/979614783.py:6: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  data = dataset.data



TIMING SUMMARY
Stage                                        Time (s)     Memory (MB)
----------------------------------------------------------------------
3a. Build incidence tensors                     0.002            +0.0
3b. 2-path motifs (hh, tt, ht)                  0.007            +0.0
3c. 3-path prep (build adjacency)               0.369           +53.6
3d. 3-path motifs (tfh, tft, hft, hfh)         49.363         +6646.5
3e. Combine hypergraph                          0.031            +0.0
----------------------------------------------------------------------
TOTAL                                          49.773         +6700.2


In [ ]:
# @title 9. Final Statistics

print("\n" + "=" * 70)
print("FINAL STATISTICS")
print("=" * 70)

print(f"\nDataset: WN18RR")
print(f"Nodes: {stats['num_nodes']}")
print(f"Edges (bidirectional): {stats['num_edges']}")
print(f"Relations: {stats['num_relations']}")
print(f"\nSplit sizes:")
print(f"  Train: {stats['num_train']}")
print(f"  Val: {stats['num_val']}")
print(f"  Test: {stats['num_test']}")

if hasattr(train_data, 'relation_graph'):
    rg = train_data.relation_graph
    print(f"\nRelation Graph:")
    print(f"  Nodes (relations): {rg.num_nodes}")
    print(f"  Edges: {rg.edge_index.size(1)}")

if hasattr(train_data, 'relation_hypergraph'):
    hg = train_data.relation_hypergraph
    print(f"\nRelation Hypergraph:")
    print(f"  Nodes (relations): {hg.num_nodes}")
    print(f"  Hyperedges: {hg.edge_index.size(1)}")
    print(f"  Relation types: {hg.num_relations}")

print("\n" + "=" * 70)
print("Done!")
print("=" * 70)

# Part 2: Testing Sampled Hypergraph Construction

Now we test the `build_relation_hypergraph_sampled` function with different sampling probabilities.

In [7]:
# @title 10. Setup: Clone Fork and Apply Patches

import os
import subprocess

# Configuration
GITHUB_USER = 'atryt0ne'
UPSTREAM_REPO = 'HxyScotthuang/MOTIF'
FORK_REPO = f'{GITHUB_USER}/MOTIF'

# Get token from Colab secrets
from google.colab import userdata
GH_TOKEN = userdata.get('gh')

# Configure git
!git config --global user.email "colab@motif.dev"
!git config --global user.name "Colab MOTIF"

# Set up authenticated URL
auth_url = f'https://{GH_TOKEN}@github.com/{FORK_REPO}.git'

# Check if fork exists, if not create it
print("Checking if fork exists...")
result = !curl -s -o /dev/null -w '%{http_code}' -H 'Authorization: token {GH_TOKEN}' https://api.github.com/repos/{FORK_REPO}
if result[0] == '404':
    print(f"Creating fork {FORK_REPO}...")
    !curl -s -X POST -H 'Authorization: token {GH_TOKEN}' https://api.github.com/repos/{UPSTREAM_REPO}/forks
    print("Fork created! Waiting for it to be ready...")
    import time
    time.sleep(5)
else:
    print(f"Fork {FORK_REPO} already exists")

# Clone the fork
print(f"\nCloning fork from {FORK_REPO}...")
!rm -rf /content/MOTIF
!git clone {auth_url} /content/MOTIF

# Add upstream as remote
!cd /content/MOTIF && git remote add upstream https://github.com/{UPSTREAM_REPO}.git

print(f"\nCloned {FORK_REPO} successfully!")

MOTIF repository already exists!


In [ ]:
# @title 10.5 Apply Patches and Commit to Fork

import torch

# ===== Patch tasks.py =====
print("Patching tasks.py...")
with open('/content/MOTIF/motif/tasks.py', 'r') as f:
    content = f.read()

# Patch 1: build_relation_graph device detection
old_pattern = '''def build_relation_graph(graph):
    # expect the graph is already with inverse edges

    edge_index, edge_type = graph.edge_index, graph.edge_type
    num_nodes, num_rels = graph.num_nodes, graph.num_relations
    if isinstance(edge_index, torch.Tensor):
        device = edge_index.device
    else:
        device = torch.device("cpu")'''

new_pattern = '''def build_relation_graph(graph):
    # expect the graph is already with inverse edges

    edge_index, edge_type = graph.edge_index, graph.edge_type
    num_nodes, num_rels = graph.num_nodes, graph.num_relations
    
    # Safely get device - handle PyG Data objects
    if torch.cuda.is_available() and hasattr(edge_index, 'is_cuda') and edge_index.is_cuda:
        device = edge_index.device
    else:
        device = torch.device("cpu")'''

content = content.replace(old_pattern, new_pattern)

# Add our new functions if they don't exist
if 'def sample_sparse_coo' not in content:
    print("Adding sample_sparse_coo and build_relation_hypergraph_sampled...")
    # Read the functions from local file
    sample_func = '''\n\ndef sample_sparse_coo(sparse_tensor, sample_prob, device):
    """Sample entries from a sparse COO tensor with probability sample_prob."""
    if sample_prob >= 1.0:
        return sparse_tensor.coalesce()
    
    indices = sparse_tensor.coalesce().indices()
    values = sparse_tensor.coalesce().values()
    num_entries = indices.size(1)
    
    if num_entries == 0:
        return sparse_tensor.coalesce()
    
    mask = torch.rand(num_entries, device=device) < sample_prob
    sampled_indices = indices[:, mask]
    sampled_values = values[mask]
    
    if sampled_indices.size(1) == 0:
        return torch.sparse_coo_tensor(
            torch.empty((indices.size(0), 0), dtype=torch.long, device=device),
            torch.empty(0, dtype=values.dtype, device=device),
            sparse_tensor.size(),
        ).coalesce()
    
    return torch.sparse_coo_tensor(
        sampled_indices, sampled_values, sparse_tensor.size()
    ).coalesce()


'''
    content = content + sample_func

with open('/content/MOTIF/motif/tasks.py', 'w') as f:
    f.write(content)

# ===== Patch layers.py =====
print("Patching layers.py...")
with open('/content/MOTIF/motif/layers.py', 'r') as f:
    layers_content = f.read()

layers_content = layers_content.replace('use_triton=True', 'use_triton=False')

with open('/content/MOTIF/motif/layers.py', 'w') as f:
    f.write(layers_content)

# ===== Commit and push =====
print("\nCommitting changes...")
!cd /content/MOTIF && git add motif/tasks.py motif/layers.py
!cd /content/MOTIF && git commit -m "Fix device detection and disable Triton kernels for Colab compatibility" || echo "Nothing to commit"

print("\nPushing to fork...")
from google.colab import userdata
GH_TOKEN = userdata.get('gh')
!cd /content/MOTIF && git push https://{GH_TOKEN}@github.com/{FORK_REPO}.git main || echo "Push failed or no changes"

print("\n✓ Patches applied and pushed to fork!")

In [ ]:
# @title 11. Test Sampled Hypergraph Construction

In [9]:
# @title 11. Test Sampled Hypergraph Construction

import sys
sys.path.insert(0, '/content/MOTIF')

from motif.tasks import build_relation_hypergraph_sampled

print("=" * 70)
print("SAMPLED HYPERGRAPH CONSTRUCTION TEST")
print("=" * 70)

# Test different sampling probabilities
sample_probs = [1.0, 0.5, 0.25, 0.1]

results = []

for p in sample_probs:
    print(f"\nTesting sample_prob={p}...")

    # Load fresh data
    train_data_p, _, _, _ = load_wn18rr(root, device)
    train_data_p = build_relation_graph(train_data_p)

    # Time the sampled hypergraph construction
    sample_timer = Timer()
    with sample_timer.time(f"sample_prob={p}"):
        train_data_p = build_relation_hypergraph_sampled(
            train_data_p,
            sample_prob=p,
            seed=42
        )

    # Get statistics
    hg = train_data_p.relation_hypergraph
    num_edges = hg.edge_index.size(1)
    type_counts = torch.bincount(hg.edge_type, minlength=7)

    results.append({
        'sample_prob': p,
        'time_s': sample_timer.results[f'sample_prob={p}']['time_s'],
        'memory_mb': sample_timer.results[f'sample_prob={p}']['memory_mb'],
        'total_edges': num_edges,
        'type_counts': type_counts.tolist()
    })

    print(f"  Time: {results[-1]['time_s']:.2f}s")
    print(f"  Memory: {results[-1]['memory_mb']:+.1f} MB")
    print(f"  Total edges: {num_edges:,}")
    print(f"  Edge types: {type_counts.tolist()}")

SAMPLED HYPERGRAPH CONSTRUCTION TEST

Testing sample_prob=1.0...


/tmp/ipykernel_16931/979614783.py:6: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  data = dataset.data


  Time: 0.34s
  Memory: +77.2 MB
  Total edges: 18,652
  Edge types: [380, 380, 380, 4378, 4378, 4378, 4378]

Testing sample_prob=0.5...
  Time: 0.10s
  Memory: +0.9 MB
  Total edges: 9,361
  Edge types: [197, 195, 172, 2183, 2232, 2156, 2226]

Testing sample_prob=0.25...
  Time: 0.06s
  Memory: +0.0 MB
  Total edges: 4,670
  Edge types: [91, 99, 81, 1042, 1106, 1093, 1158]

Testing sample_prob=0.1...
  Time: 0.05s
  Memory: +0.0 MB
  Total edges: 1,876
  Edge types: [42, 38, 39, 422, 441, 431, 463]


In [10]:
# @title 12. Comparison Summary Table

print("\n" + "=" * 80)
print("SAMPLED HYPERGRAPH COMPARISON")
print("=" * 80)

print(f"{'sample_prob':<12} {'Time (s)':>10} {'Memory (MB)':>12} {'Total Edges':>15} {'Speedup':>10}")
print("-" * 80)

baseline_time = results[0]['time_s']

for r in results:
    speedup = baseline_time / r['time_s'] if r['time_s'] > 0 else float('inf')
    print(f"{r['sample_prob']:<12.2f} {r['time_s']:>10.2f} {r['memory_mb']:>+12.1f} {r['total_edges']:>15,} {speedup:>10.1f}x")

print("=" * 80)

print("\nEdge counts per type:")
type_names = ['hh', 'tt', 'ht', 'tfh', 'tft', 'hft', 'hfh']
print(f"{'sample_prob':<12} " + " ".join([f"{name:>10}" for name in type_names]))
print("-" * 80)
for r in results:
    counts = r['type_counts']
    print(f"{r['sample_prob']:<12.2f} " + " ".join([f"{c:>10,}" for c in counts]))
print("=" * 80)


SAMPLED HYPERGRAPH COMPARISON
sample_prob    Time (s)  Memory (MB)     Total Edges    Speedup
--------------------------------------------------------------------------------
1.00               0.34        +77.2          18,652        1.0x
0.50               0.10         +0.9           9,361        3.3x
0.25               0.06         +0.0           4,670        6.0x
0.10               0.05         +0.0           1,876        7.1x

Edge counts per type:
sample_prob          hh         tt         ht        tfh        tft        hft        hfh
--------------------------------------------------------------------------------
1.00                380        380        380      4,378      4,378      4,378      4,378
0.50                197        195        172      2,183      2,232      2,156      2,226
0.25                 91         99         81      1,042      1,106      1,093      1,158
0.10                 42         38         39        422        441        431        463


In [11]:
# @title 13. Verify Proportional Reduction

print("\n" + "=" * 70)
print("PROPORTIONAL REDUCTION VERIFICATION")
print("=" * 70)

baseline = results[0]
print(f"\nBaseline (sample_prob=1.0): {baseline['total_edges']:,} edges")
print("\nExpected vs Actual edge counts:")
print(f"{'sample_prob':<12} {'Expected':>15} {'Actual':>15} {'Ratio':>10}")
print("-" * 55)

for r in results[1:]:
    expected = int(baseline['total_edges'] * r['sample_prob'])
    actual = r['total_edges']
    ratio = actual / expected if expected > 0 else 0
    print(f"{r['sample_prob']:<12.2f} {expected:>15,} {actual:>15,} {ratio:>10.2f}")

print("\nNote: Ratio should be close to 1.0 for proportional stratified sampling")
print("=" * 70)


PROPORTIONAL REDUCTION VERIFICATION

Baseline (sample_prob=1.0): 18,652 edges

Expected vs Actual edge counts:
sample_prob         Expected          Actual      Ratio
-------------------------------------------------------
0.50                   9,326           9,361       1.00
0.25                   4,663           4,670       1.00
0.10                   1,865           1,876       1.01

Note: Ratio should be close to 1.0 for proportional stratified sampling


# Part 3: Testing Python Loop Implementation with Sampling (96M edges baseline)

In [ ]:
# @title 14. Test Python Loop Sampled Hypergraph (large scale)

print("=" * 70)
print("PYTHON LOOP SAMPLED HYPERGRAPH TEST (96M edges baseline)")
print("=" * 70)
print("\nNote: This will test sampling on the Python loop implementation")
print("that produces ~96M edges (vs repo's ~18K edges)")

# Test different sampling probabilities - use smaller values to avoid OOM
sample_probs_python = [1.0, 0.25, 0.1, 0.05]

python_results = []

for p in sample_probs_python:
    print(f"\nTesting sample_prob={p}...")
    
    # Load fresh data
    train_data_py, _, _, _ = load_wn18rr(root, device)
    train_data_py = build_relation_graph(train_data_py)
    
    # Time the sampled hypergraph construction
    py_timer = Timer()
    with py_timer.time(f"sample_prob={p}"):
        train_data_py = build_relation_hypergraph_sampled_python(
            train_data_py, 
            sample_prob=p, 
            seed=42
        )
    
    # Get statistics
    hg = train_data_py.relation_hypergraph
    num_edges = hg.edge_index.size(1)
    type_counts = torch.bincount(hg.edge_type, minlength=7)
    
    python_results.append({
        'sample_prob': p,
        'time_s': py_timer.results[f'sample_prob={p}']['time_s'],
        'memory_mb': py_timer.results[f'sample_prob={p}']['memory_mb'],
        'total_edges': num_edges,
        'type_counts': type_counts.tolist()
    })
    
    print(f"  Time: {python_results[-1]['time_s']:.2f}s")
    print(f"  Memory: {python_results[-1]['memory_mb']:+.1f} MB")
    print(f"  Total edges: {num_edges:,}")
    
    # Clean up to free memory
    del train_data_py
    if device == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# @title 15. Python Loop Results Summary

print("\n" + "=" * 80)
print("PYTHON LOOP SAMPLED HYPERGRAPH COMPARISON (96M baseline)")
print("=" * 80)

print(f"{'sample_prob':<12} {'Time (s)':>10} {'Memory (MB)':>12} {'Total Edges':>15} {'Speedup':>10}")
print("-" * 80)

py_baseline_time = python_results[0]['time_s']

for r in python_results:
    speedup = py_baseline_time / r['time_s'] if r['time_s'] > 0 else float('inf')
    print(f"{r['sample_prob']:<12.2f} {r['time_s']:>10.2f} {r['memory_mb']:>+12.1f} {r['total_edges']:>15,} {speedup:>10.1f}x")

print("=" * 80)

print("\nEdge counts per type:")
type_names = ['hh', 'tt', 'ht', 'tfh', 'tft', 'hft', 'hfh']
print(f"{'sample_prob':<12} " + " ".join([f"{name:>10}" for name in type_names]))
print("-" * 80)
for r in python_results:
    counts = r['type_counts']
    print(f"{r['sample_prob']:<12.2f} " + " ".join([f"{c:>10,}" if c < 1000000 else f"{c/1e6:>9.1f}M" for c in counts]))
print("=" * 80)

In [ ]:
# @title 16. Verify Proportional Reduction for Python Loop

print("\n" + "=" * 70)
print("PROPORTIONAL REDUCTION VERIFICATION (Python Loop)")
print("=" * 70)

py_baseline = python_results[0]
print(f"\nBaseline (sample_prob=1.0): {py_baseline['total_edges']:,} edges")
print("\nExpected vs Actual edge counts:")
print(f"{'sample_prob':<12} {'Expected':>15} {'Actual':>15} {'Ratio':>10}")
print("-" * 55)

for r in python_results[1:]:
    expected = int(py_baseline['total_edges'] * r['sample_prob'])
    actual = r['total_edges']
    ratio = actual / expected if expected > 0 else 0
    print(f"{r['sample_prob']:<12.2f} {expected:>15,} {actual:>15,} {ratio:>10.2f}")

print("\nNote: Ratio should be close to 1.0 for proportional stratified sampling")
print("=" * 70)

# Part 4: Model Training with Sampled Hypergraphs

Now we train the MOTIF model with different sampling rates and compare MRR.

In [ ]:
# @title 17. Setup Training Functions

import sys
sys.path.insert(0, '/content/MOTIF')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch_geometric.data import Data
import math
import random

from motif.models import MOTIF
from motif.tasks import negative_sampling, all_negative, compute_ranking, strict_negative_mask, build_relation_graph
from motif.tasks import build_relation_hypergraph_sampled

def prepare_data_with_sampling(root, sample_prob=1.0, device='cuda', seed=42):
    """Load WN18RR and prepare data with sampled hypergraph."""
    random.seed(seed)
    torch.manual_seed(seed)
    
    # Load dataset
    dataset = WordNet18RR(root=root)
    data = dataset.data
    
    num_nodes = int(data.edge_index.max()) + 1
    num_relations = int(data.edge_type.max()) + 1
    
    train_mask = data.train_mask if hasattr(data, 'train_mask') else torch.ones(data.edge_index.size(1), dtype=torch.bool)
    val_mask = getattr(data, 'val_mask', torch.zeros(data.edge_index.size(1), dtype=torch.bool))
    test_mask = getattr(data, 'test_mask', torch.zeros(data.edge_index.size(1), dtype=torch.bool))
    
    # Training graph: edges with inverses
    train_idx = data.edge_index[:, train_mask]
    train_type = data.edge_type[train_mask]
    
    edge_index = torch.cat([train_idx, train_idx.flip(0)], dim=-1)
    edge_type = torch.cat([train_type, train_type + num_relations])
    
    train_data = Data(
        edge_index=edge_index,
        edge_type=edge_type,
        num_nodes=num_nodes,
        target_edge_index=train_idx,
        target_edge_type=train_type,
        num_relations=num_relations * 2
    ).to(device)
    
    # Build relation graph
    train_data = build_relation_graph(train_data)
    
    # Build relation hypergraph with sampling
    train_data = build_relation_hypergraph_sampled(
        train_data, 
        sample_prob=sample_prob, 
        seed=seed
    )
    
    # Validation data
    val_idx = data.edge_index[:, val_mask]
    val_type = data.edge_type[val_mask]
    val_data = Data(
        edge_index=edge_index,
        edge_type=edge_type,
        num_nodes=num_nodes,
        target_edge_index=val_idx,
        target_edge_type=val_type,
        num_relations=num_relations * 2
    ).to(device)
    val_data.relation_graph = train_data.relation_graph
    val_data.relation_hypergraph = train_data.relation_hypergraph
    
    # Test data
    test_idx = data.edge_index[:, test_mask]
    test_type = data.edge_type[test_mask]
    test_data = Data(
        edge_index=edge_index,
        edge_type=edge_type,
        num_nodes=num_nodes,
        target_edge_index=test_idx,
        target_edge_type=test_type,
        num_relations=num_relations * 2
    ).to(device)
    test_data.relation_graph = train_data.relation_graph
    test_data.relation_hypergraph = train_data.relation_hypergraph
    
    return train_data, val_data, test_data

def create_model(device='cuda'):
    """Create MOTIF model with default config."""
    # Disable use_triton to avoid CUDA kernel compilation issues on Colab
    model = MOTIF(
        rel_model_cfg={'input_dim': 64, 'hidden_dims': [64, 64, 64, 64, 64, 64], 
                      'short_cut': True, 'aggregate_func': 'sum', 'use_triton': False},
        entity_model_cfg={'input_dim': 64, 'hidden_dims': [64, 64, 64, 64, 64, 64],
                         'message_func': 'distmult', 'aggregate_func': 'sum',
                         'short_cut': True, 'layer_norm': True, 'use_triton': False}
    )
    return model.to(device)

@torch.no_grad()
def evaluate(model, data, filtered_data=None, batch_size=64, device='cuda'):
    """Evaluate model on validation/test set."""
    model.eval()
    
    triplets = torch.cat([data.target_edge_index, data.target_edge_type.unsqueeze(0)]).t()
    loader = DataLoader(triplets, batch_size=batch_size, shuffle=False)
    
    rankings = []
    for batch in loader:
        batch = batch.to(device)
        t_batch, h_batch = all_negative(data, batch)
        
        t_pred = model(data, t_batch)
        h_pred = model(data, h_batch)
        
        if filtered_data is None:
            t_mask, h_mask = strict_negative_mask(data, batch)
        else:
            t_mask, h_mask = strict_negative_mask(filtered_data, batch)
        
        pos_h, pos_t, _ = batch.t()
        t_ranking = compute_ranking(t_pred, pos_t, t_mask)
        h_ranking = compute_ranking(h_pred, pos_h, h_mask)
        
        rankings.extend([t_ranking, h_ranking])
    
    ranking = torch.cat(rankings)
    mrr = (1 / ranking.float()).mean().item()
    mr = ranking.float().mean().item()
    hits1 = (ranking <= 1).float().mean().item()
    hits3 = (ranking <= 3).float().mean().item()
    hits10 = (ranking <= 10).float().mean().item()
    
    return {'mrr': mrr, 'mr': mr, 'hits@1': hits1, 'hits@3': hits3, 'hits@10': hits10}

In [ ]:
# @title 18. Training Loop

def train_epoch(model, data, optimizer, batch_size=64, num_negative=256, device='cuda'):
    """Train for one epoch."""
    model.train()
    
    triplets = torch.cat([data.target_edge_index, data.target_edge_type.unsqueeze(0)]).t()
    loader = DataLoader(triplets, batch_size=batch_size, shuffle=True)
    
    total_loss = 0
    num_batches = 0
    
    for batch in loader:
        batch = batch.to(device)
        batch = negative_sampling(data, batch, num_negative, strict=True)
        
        pred = model(data, batch)
        target = torch.zeros_like(pred)
        target[:, 0] = 1
        
        loss = F.binary_cross_entropy_with_logits(pred, target, reduction='none')
        neg_weight = torch.ones_like(pred)
        neg_weight[:, 1:] = 1 / num_negative
        loss = (loss * neg_weight).sum(dim=-1) / neg_weight.sum(dim=-1)
        loss = loss.mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches

def train_model(train_data, val_data, num_epochs=10, batch_size=64, lr=1e-3, device='cuda', verbose=True):
    """Full training loop."""
    model = create_model(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    best_mrr = 0
    best_epoch = 0
    
    for epoch in range(num_epochs):
        loss = train_epoch(model, train_data, optimizer, batch_size=batch_size, device=device)
        
        if (epoch + 1) % 2 == 0 or epoch == 0:
            metrics = evaluate(model, val_data, batch_size=batch_size, device=device)
            if metrics['mrr'] > best_mrr:
                best_mrr = metrics['mrr']
                best_epoch = epoch + 1
            
            if verbose:
                print(f"Epoch {epoch+1:2d} | Loss: {loss:.4f} | Val MRR: {metrics['mrr']:.4f} | Hits@10: {metrics['hits@10']:.4f}")
    
    return model, {'best_mrr': best_mrr, 'best_epoch': best_epoch}

In [ ]:
# @title 19. Run Experiments with Different Sample Rates

print("=" * 70)
print("TRAINING EXPERIMENTS: Effect of Hypergraph Sampling on MRR")
print("=" * 70)
print("\nNote: Using smaller epochs for quick experiments.")
print("For real experiments, use num_epochs=10-20")

# Experiment settings
sample_probs_to_test = [1.0, 0.5, 0.25]
num_epochs = 4  # Use more epochs for real experiments
batch_size = 32

experiment_results = []

for p in sample_probs_to_test:
    print(f"\n{'='*70}")
    print(f"Experiment: sample_prob = {p}")
    print(f"{'='*70}")
    
    # Prepare data with this sampling rate
    print("Preparing data...")
    train_data, val_data, test_data = prepare_data_with_sampling(
        root='/content/wn18rr',
        sample_prob=p,
        device=device,
        seed=42
    )
    
    hg = train_data.relation_hypergraph
    print(f"Hypergraph edges: {hg.edge_index.size(1):,}")
    
    # Train model
    print("\nTraining...")
    model, train_metrics = train_model(
        train_data, val_data,
        num_epochs=num_epochs,
        batch_size=batch_size,
        device=device,
        verbose=True
    )
    
    # Final evaluation
    print("\nFinal evaluation on validation set...")
    val_metrics = evaluate(model, val_data, batch_size=batch_size, device=device)
    
    experiment_results.append({
        'sample_prob': p,
        'hypergraph_edges': hg.edge_index.size(1),
        'train_best_mrr': train_metrics['best_mrr'],
        'val_mrr': val_metrics['mrr'],
        'val_hits@1': val_metrics['hits@1'],
        'val_hits@10': val_metrics['hits@10'],
        'best_epoch': train_metrics['best_epoch']
    })
    
    print(f"\nResults for sample_prob={p}:")
    print(f"  Val MRR: {val_metrics['mrr']:.4f}")
    print(f"  Val Hits@1: {val_metrics['hits@1']:.4f}")
    print(f"  Val Hits@10: {val_metrics['hits@10']:.4f}")
    
    # Cleanup
    del model, train_data, val_data, test_data
    torch.cuda.empty_cache()

In [ ]:
# @title 20. Experiment Results Summary

print("\n" + "=" * 80)
print("EXPERIMENT RESULTS SUMMARY")
print("=" * 80)

print(f"{'sample_prob':<12} {'Edges':>12} {'Val MRR':>10} {'Hits@1':>10} {'Hits@10':>10} {'Rel. MRR':>10}")
print("-" * 80)

baseline_mrr = experiment_results[0]['val_mrr']

for r in experiment_results:
    rel_mrr = r['val_mrr'] / baseline_mrr * 100
    edges_str = f"{r['hypergraph_edges']:,}" if r['hypergraph_edges'] < 1000000 else f"{r['hypergraph_edges']/1e6:.1f}M"
    print(f"{r['sample_prob']:<12.2f} {edges_str:>12} {r['val_mrr']:>10.4f} {r['val_hits@1']:>10.4f} {r['val_hits@10']:>10.4f} {rel_mrr:>9.1f}%")

print("=" * 80)

print("\nKey Insight:")
if experiment_results[-1]['val_mrr'] / baseline_mrr > 0.90:
    print("  ✅ Sampling preserves >90% accuracy with substantial speedup!")
else:
    print("  ⚠️ Accuracy drop exceeds 10% - consider larger sample_prob")

print("\nSpeedup Analysis:")
for r in experiment_results:
    baseline_edges = experiment_results[0]['hypergraph_edges']
    if r['sample_prob'] < 1.0:
        speedup = baseline_edges / r['hypergraph_edges']
        print(f"  sample_prob={r['sample_prob']:.2f}: ~{speedup:.1f}x faster preprocessing")